**CI twin of `ch16-the-math-of-one-neuron.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

x = np.array([2.0, 3.0])
w = np.array([0.5, -1.0])
b = 0.5
y = 1.0

z = float(w @ x + b)          # weighted sum (dot product + bias)
a = float(sigmoid(z))         # activation, squashed into 0..1
loss = float((a - y) ** 2)    # squared error: how wrong we were

print("z (weighted sum) =", round(z, 4))
print("a (prediction)   =", round(a, 6))
print("loss             =", round(loss, 6))

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

x = np.array([2.0, 3.0])
w = np.array([0.5, -1.0])
b = 0.5
y = 1.0

# forward again, so this cell stands alone
z = w @ x + b
a = sigmoid(z)

# backward: chain rule, link by link
dL_da = 2 * (a - y)        # derivative of (a - y)^2
da_dz = a * (1 - a)        # derivative of the sigmoid
dL_dz = dL_da * da_dz      # combine the first two links

grad_w = dL_dz * x         # dz/dw_i = x_i  (fan out to each weight)
grad_b = dL_dz * 1.0       # dz/db   = 1

print("dL_dz   =", round(float(dL_dz), 6))
print("grad_w  =", [round(float(g), 6) for g in grad_w])
print("grad_b  =", round(float(grad_b), 6))

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

x = np.array([2.0, 3.0])
w = np.array([0.5, -1.0])
b = 0.5
y = 1.0

def loss_at(w, b):
    return float((sigmoid(w @ x + b) - y) ** 2)

# analytic gradient (from the backward pass above)
a = sigmoid(w @ x + b)
dL_dz = 2 * (a - y) * a * (1 - a)
analytic = np.array([dL_dz * x[0], dL_dz * x[1], dL_dz])

# numeric gradient by finite differences
eps = 1e-5
numeric = []
for i in range(2):                       # the two weights
    wp = w.copy(); wp[i] += eps
    wm = w.copy(); wm[i] -= eps
    numeric.append((loss_at(wp, b) - loss_at(wm, b)) / (2 * eps))
numeric.append((loss_at(w, b + eps) - loss_at(w, b - eps)) / (2 * eps))  # the bias
numeric = np.array(numeric)

max_error = float(np.max(np.abs(analytic - numeric)))
print("analytic =", [round(float(g), 6) for g in analytic])
print("numeric  =", [round(float(g), 6) for g in numeric])
print("max error = {:.2e}".format(max_error))

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

run_tests([
    ("zero maps to a half", float(sigmoid(0)), 0.5),
    ("sigmoid(2)", round(float(sigmoid(2)), 4), 0.8808),
])

In [ ]:
import numpy as np

def neuron_forward(x, w, b):
    z = np.array(w) @ np.array(x) + b
    return 1.0 / (1.0 + np.exp(-z))

run_tests([
    ("our worked example", round(float(neuron_forward([2, 3], [0.5, -1.0], 0.5)), 4), 0.1824),
    ("inputs of one", round(float(neuron_forward([1, 1], [1, 1], 0.0)), 4), 0.8808),
])

In [ ]:
def dL_dz(a, y):
    return 2 * (a - y) * a * (1 - a)

run_tests([
    ("under-predicting", round(dL_dz(0.5, 1.0), 4), -0.25),
    ("over-predicting", round(dL_dz(0.5, 0.0), 4), 0.25),
])